# Training Pipeline — QCNN

A **Quantum Convolutional Neural Network**, ported from
[`takh04/QCNN`](https://github.com/takh04/QCNN) (Hur, Kim & Park, *Quantum convolutional neural
network for classical data classification*, 2022). Unlike the other notebooks in this folder,
there is no PyTorch backbone to configure through `timm` — the original is a PennyLane circuit
simulated one 8-qubit statevector at a time, so the load-bearing pieces are ported as plain
`torch` tensor ops, the same departure [`vae/notebooks/quantum/qvae.ipynb`](../../../vae/notebooks/quantum/qvae.ipynb)
takes from its own Qiskit source.

Trained and evaluated on **BreastMNIST**, matching
[`cnn/notebooks/classical/resnet50.ipynb`](../../notebooks/classical/resnet50.ipynb) and its
siblings — same dataset, same [`_handlers/evaluation.py::evaluate_all_metrics`](../../../_handlers/evaluation.py)
metric suite, so the quantum and classical runs are directly comparable.

**What's ported, in [`cnn/models/qcnn.py`](../../models/qcnn.py):**
- `U_SU4` (`unitary.py`) — the 15-parameter general two-qubit convolution ansatz the original's
  benchmarking run uses, built here as a dense `2^8 x 2^8` unitary matrix instead of a PennyLane
  sub-circuit.
- `Pooling_ansatz1` (`unitary.py`) — the only pooling ansatz the original ever calls
  (`CRZ; PauliX; CRX`). Pooling does **not** trace anything out — the discarded wire is simply
  never addressed again — so this port never needs a partial trace, unlike `qvae.ipynb`'s
  trash-qubit bottleneck.
- `QCNN_structure` (`QCNN_circuit.py`) — 3 conv+pool stages reducing 8 qubits to 4, to 2, to 1,
  each conv layer *reusing one shared parameter vector* across every wire pair (the circuit's
  analogue of a classical conv kernel), reading out the final surviving qubit (wire 4).

**What's replaced:** the original loops over samples one at a time and optimizes with
`NesterovMomentumOptimizer` via PennyLane's own autograd (parameter-shift under the hood). Since
every op here (unitary construction, matrix multiply, `|amplitude|^2`) is an ordinary
differentiable `torch` op, this notebook instead batches the whole circuit over the batch
dimension and trains with plain `torch.autograd` + Adam.

Data feeds through the amplitude embedding (`data.py`'s `'resize256'` mode) and the loss shape
(cross-entropy on `[P(0), P(1)]`) matches `data.py` / `Training.py` exactly — see
[`cnn/handlers/qcnn.py`](../../handlers/qcnn.py). Final evaluation, however, swaps the original's
own `Benchmarking.py::accuracy_test` decoding for the shared `evaluate_all_metrics` helper, so
the reported numbers are the medmnist Evaluator AUC/ACC plus the full scikit-learn suite —
precision, recall/sensitivity, specificity, F1, confusion matrix, per-class report — exactly like
every classical notebook in this repo.

A GPU helps but is not required — the whole circuit is one `256x256` complex matrix multiply per
batch, tiny by CNN standards. `build_model` below does not require CUDA.

## Install Requirements

What the pipeline imports: `torch` for the model/training loop, `medmnist` for BreastMNIST,
`scikit-learn` for the metrics, plus `tqdm`. No `pennylane` — the circuit is reimplemented as
dense `torch` matrices, so the original's quantum-simulation dependency is never installed.

In [1]:
!nvidia-smi

Mon Jul 27 21:32:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 107.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 114.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/2

In [3]:
!pip install medmnist==3.0.2 scikit-learn tqdm requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 11.9 MB/s eta 0:00:00


## Get the survey code

The model and handlers live in this repository's `survey/src/cnn` and `survey/src/_handlers`
packages, so the notebook needs a checkout of it. On Colab it clones into
`/content/quantum-quantization`, or `git pull --ff-only`s that directory if it is already there —
so re-running the cell after a push picks up the new code. Run locally, the notebook already sits
inside the repo, so `find_src` climbs to `survey/src` and git is never touched (your working tree
is left alone).

The branch is chosen by *environment*, not by working directory: the imports cell `os.chdir`s into
the checkout, so a cwd-based test would find `cnn` on every re-run and silently skip the pull.

If the repository is private the anonymous clone fails with an authentication error; use a token
URL instead — `REPO_URL = 'https://<GITHUB_TOKEN>@github.com/alexandrachirita98/quantum-quantization.git'`.

In [4]:
import pathlib
import subprocess
import sys

REPO_URL = 'https://github.com/alexandrachirita98/quantum-quantization.git'

# NB: key off the environment, not the working directory. `os.chdir` in the imports cell moves the
# cwd *inside* the checkout, so a cwd-based test would report "already have the code" on every
# re-run and silently skip the pull.
IN_COLAB = 'google.colab' in sys.modules or pathlib.Path('/content').is_dir()
CLONE_DIR = pathlib.Path('/content/quantum-quantization')   # where the Colab checkout goes


def find_src(start):
    """Climb from `start` looking for the survey `src/` root — the directory holding `cnn`."""
    for p in [pathlib.Path(start), *pathlib.Path(start).parents]:
        if (p / 'cnn' / 'models' / 'qcnn.py').is_file():
            return p
    return None


if IN_COLAB:
    if CLONE_DIR.exists():                                  # refresh whatever was cloned earlier
        subprocess.run(['git', 'pull', '--ff-only'], cwd=str(CLONE_DIR), check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(CLONE_DIR)], check=True)
    SRC = CLONE_DIR / 'survey' / 'src'
else:                                                       # local: the notebook lives in the repo
    SRC = find_src(pathlib.Path.cwd())

assert SRC is not None and (SRC / 'cnn' / 'models' / 'qcnn.py').is_file(), f'qcnn.py not found under {SRC}'
print('survey src:', SRC)

survey src: /content/quantum-quantization/survey/src


## Imports

`QCNN` and the training helpers come from the survey's `cnn` package; `evaluate_all_metrics`
(the same all-metrics evaluator every classical notebook uses) comes from `_handlers`. `SRC` from
the previous cell goes on `sys.path`, and `os.chdir` moves into it so `./data` (the shared
[`src/data`](../../../data) folder) is where `medmnist` downloads BreastMNIST.

In [5]:
import os
import sys

import torch
import torch.optim as optim
from medmnist import BreastMNIST

# import the pipeline from the survey `src/` root, and work from there so './data' resolves inside it
sys.path.insert(0, str(SRC))
os.chdir(SRC)
print('working directory:', os.getcwd())

# drop cached modules, so a `git pull` above is actually reflected on a re-run
for _m in [m for m in list(sys.modules) if m in ('cnn', '_handlers') or m.startswith(('cnn.', '_handlers.'))]:
    del sys.modules[_m]

from cnn.models.qcnn import QCNN
from cnn.handlers.qcnn import to_amplitude_states, train_qcnn, RawImageDataset, QCNNLogits
from _handlers.evaluation import evaluate_all_metrics

working directory: /content/quantum-quantization/survey/src


## Configuration

`result.py`'s shipped config, kept as a plain namespace instead of `argparse`: binary classes,
the `'resize256'` amplitude embedding, `U_SU4` convolutions, cross-entropy cost. `steps` counts
optimizer **iterations**, not epochs — each step draws one fresh minibatch sampled *with
replacement*, matching `Training.py::circuit_training`.

**Dataset** — `breastmnist`, the same medmnist flag
[`resnet50.ipynb`](../../notebooks/classical/resnet50.ipynb) trains on, at its native 28x28
resolution (the amplitude embedding needs `28*28 = 784` pixels to resize down to `256 = 2^8`
amplitudes — there is no `_224` variant to fetch here).

In [6]:
from types import SimpleNamespace

args = SimpleNamespace(
    dataset='breastmnist',                            # a medmnist flag, native 28x28 resolution
    lr=0.01,                                          # matches the original's Nesterov stepsize
    steps=200,                                        # optimizer iterations, not epochs
    batch_size=25,                                    # sampled with replacement each step
)

## Device

In [7]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using {} device.".format(device))

Using cuda:0 device.


## Dataset

BreastMNIST (malignant vs. normal/benign, already binary — `label_map` is not needed here, unlike
the original repo's Fashion-MNIST class-pair filtering). Each 28x28 image is bilinear-resized to a
256 = 2^8-length vector and L2-normalized (`to_amplitude_states`, the `'resize256'` mode feeding
`AmplitudeEmbedding` in the original) so it is a valid 8-qubit statevector.

`train_raw`/`test_raw` also back a `RawImageDataset` each, used only by the final evaluation cell
below — `evaluate_all_metrics` expects a `Dataset` of raw `(image, label)` pairs and applies the
amplitude embedding itself (via `QCNNLogits`), the same way every classical notebook here hands it
a `Dataset` of already-transformed images.

In [8]:
import os

os.makedirs('./data', exist_ok=True)   # BreastMNIST() checks root exists *before* downloading

train_raw = BreastMNIST(split='train', download=True, root='./data')
test_raw = BreastMNIST(split='test', download=True, root='./data')

train_x = to_amplitude_states(torch.from_numpy(train_raw.imgs))
train_y = torch.from_numpy(train_raw.labels).reshape(-1).long()
test_x = to_amplitude_states(torch.from_numpy(test_raw.imgs))
test_y = torch.from_numpy(test_raw.labels).reshape(-1).long()

train_dataset = RawImageDataset(train_raw.imgs, train_raw.labels)
test_dataset = RawImageDataset(test_raw.imgs, test_raw.labels)

print('train:', train_x.shape, train_y.shape)
print('test: ', test_x.shape, test_y.shape)

100%|██████████| 560k/560k [00:00<00:00, 560kB/s]

Using downloaded and verified file: ./data/breastmnist.npz
train: torch.Size([546, 256]) torch.Size([546])
test:  torch.Size([156, 256]) torch.Size([156])


## Model

`QCNN()` builds the fixed 8-qubit conv+pool topology described above, with `float64` parameters
(unitary/amplitude math is numerically sensitive, matching `qvae.ipynb`'s own choice of dtype).
`forward` returns `[P(0), P(1)]` marginal probabilities of the final surviving qubit (wire 4),
matching `qml.probs(wires=4)` in the original's cross-entropy mode.

In [9]:
net = QCNN().to(device)
print(net)
print('trainable parameters:', sum(p.numel() for p in net.parameters()))

QCNN()
trainable parameters: 51


## Optimizer

Plain Adam in place of the original's `NesterovMomentumOptimizer` — the circuit is fully
differentiable end to end (dense unitary matmuls + `|amplitude|^2`), so ordinary backprop replaces
PennyLane's parameter-shift gradients without changing what's being optimized.

In [10]:
optimizer = optim.Adam(net.parameters(), lr=args.lr)

## Train

`train_qcnn` (from the handler) runs `args.steps` minibatch iterations — each step samples
`args.batch_size` training images with replacement, exactly `Training.py::circuit_training` — and
reports test accuracy once training finishes.

In [11]:
test_acc = train_qcnn(net, optimizer, train_x, train_y, test_x, test_y,
                      steps=args.steps, batch_size=args.batch_size, device=device)

step 10/200  loss: 0.7445  batch_acc: 0.6400
step 20/200  loss: 0.5709  batch_acc: 0.7600
step 30/200  loss: 0.4928  batch_acc: 0.8000
step 40/200  loss: 0.5081  batch_acc: 0.8000
step 50/200  loss: 0.4015  batch_acc: 0.8800
step 60/200  loss: 0.5523  batch_acc: 0.7600
step 70/200  loss: 0.5868  batch_acc: 0.6800
step 80/200  loss: 0.6190  batch_acc: 0.6000
step 90/200  loss: 0.6957  batch_acc: 0.6400
step 100/200  loss: 0.4756  batch_acc: 0.8000
step 110/200  loss: 0.5760  batch_acc: 0.6800
step 120/200  loss: 0.4679  batch_acc: 0.8000
step 130/200  loss: 0.4261  batch_acc: 0.8400
step 140/200  loss: 0.5493  batch_acc: 0.7200
step 150/200  loss: 0.5767  batch_acc: 0.7200
step 160/200  loss: 0.5190  batch_acc: 0.7600
step 170/200  loss: 0.6158  batch_acc: 0.6800
step 180/200  loss: 0.5199  batch_acc: 0.7600
step 190/200  loss: 0.5941  batch_acc: 0.6800
step 200/200  loss: 0.6359  batch_acc: 0.6000

Finished training. test_accuracy: 0.7436


## Evaluate all metrics

Same helper every classical notebook in this folder uses:
[`evaluate_all_metrics`](../../../_handlers/evaluation.py) runs the model once over one split and
prints **every** metric the training routines can produce — the medmnist Evaluator AUC/ACC plus
accuracy, weighted precision / recall (sensitivity) / F1, per-class + average specificity,
one-vs-rest AUC, the confusion matrix and a per-class report.

`QCNNLogits` adapts `QCNN` to the contract `evaluate_all_metrics` expects (raw images in, logits
out): it runs `to_amplitude_states` on each batch, feeds it through the trained circuit, and
returns `log(probs)` — `evaluate_all_metrics`'s internal `.softmax(dim=-1)` undoes the log and
recovers the exact same `[P(0), P(1)]` the circuit produced, so wrapping it changes nothing about
what gets measured. `size=28` tells the medmnist `Evaluator` to score against the native-resolution
`.npz` (the amplitude embedding never sees the `_224` variant). Set `split` to `'train'` or
`'test'`.

In [12]:
split = 'test'   # 'train' or 'test'

eval_net = QCNNLogits(net)
eval_dataset = train_dataset if split == 'train' else test_dataset
metrics = evaluate_all_metrics(eval_net, eval_dataset, args.dataset, nb_classes=2, device=device,
                               split=split, batch_size=2 * args.batch_size, size=28)

100%|██████████| 4/4 [00:00<00:00,  4.63it/s]
[medmnist Evaluator]  auc: 0.6796  acc: 0.7436

=== test metrics (2 classes) ===
accuracy            : 0.7436
overall_accuracy    : 0.7436
auc (ovr)           : 0.6796
precision (weighted): 0.8102
recall / sensitivity: 0.7436
specificity (avg)   : 0.5238
f1 (weighted)       : 0.6462

per-class specificity: ['1.000', '0.048']

confusion matrix:
[[  2  40]
 [  0 114]]

classification report:
              precision    recall  f1-score   support

           0       1.00      0.05      0.09        42
           1       0.74      1.00      0.85       114

    accuracy                           0.74       156
   macro avg       0.87      0.52      0.47       156
weighted avg       0.81      0.74      0.65       156

